# Ridge & Lasso Regression

Ridge and Lasso are **regularised** extensions of Linear Regression that add a **penalty term** to the loss function to prevent overfitting and handle multicollinearity.

---

### Ridge Regression (L2 Regularisation)
$$\text{Loss} = \text{MSE} + \alpha \sum_{i=1}^{n} \beta_i^2$$

- Shrinks coefficients towards zero but **never makes them exactly zero**.
- Useful when many features contribute modestly.

### Lasso Regression (L1 Regularisation)
$$\text{Loss} = \text{MSE} + \alpha \sum_{i=1}^{n} |\beta_i|$$

- Can **shrink coefficients to exactly zero** → automatic feature selection.
- Useful when only a few features are truly important.

| | Ridge | Lasso |
|-|-------|-------|
| Penalty | $L2$ (squared) | $L1$ (absolute) |
| Feature Selection | No | Yes |
| Handles Multicollinearity | Yes | Partial |

---

## Why Diabetes Dataset?
> The **Diabetes** dataset has 10 features that are mildly correlated with each other. This multicollinearity is exactly the condition where Ridge and Lasso shine — Ridge distributes weight across correlated features while Lasso zeros out the less important ones. The dataset is a standard benchmark specifically designed for regression regularisation comparisons.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardise features (important for regularised models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

alpha = 1.0

# Baseline
lr = LinearRegression().fit(X_train_scaled, y_train)

# Ridge
ridge = Ridge(alpha=alpha)
ridge.fit(X_train_scaled, y_train)

# Lasso
lasso = Lasso(alpha=alpha)
lasso.fit(X_train_scaled, y_train)

In [ ]:
for name, model in [('Linear (no reg)', lr), ('Ridge', ridge), ('Lasso', lasso)]:
    y_pred = model.predict(X_test_scaled)
    print(f'{name:<20} R²={r2_score(y_test, y_pred):.4f}  RMSE={np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')

# Coefficient Comparison
coef_df = pd.DataFrame({
    'Feature'       : X.columns,
    'Linear'        : lr.coef_,
    'Ridge'         : ridge.coef_,
    'Lasso'         : lasso.coef_
})
print('\nCoefficients:')
print(coef_df)

In [ ]:
# Coefficient comparison bar chart
x = np.arange(len(X.columns))
width = 0.28

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, lr.coef_,    width, label='Linear', color='steelblue')
ax.bar(x,         ridge.coef_,  width, label='Ridge',  color='darkorange')
ax.bar(x + width, lasso.coef_,  width, label='Lasso',  color='green')
ax.set_xticks(x)
ax.set_xticklabels(X.columns, rotation=30)
ax.set_ylabel('Coefficient Value')
ax.set_title('Ridge vs Lasso — Coefficient Comparison')
ax.legend()
plt.tight_layout()
plt.show()